# 02 · Embeddings와 유사도

**목표**: 텍스트를 숫자 벡터로 바꾸고 코사인 유사도로 의미상 가까운 문장을 비교합니다.

임베딩은 RAG와 grounding 검색의 기반입니다. 질문과 문서 조각을 같은 벡터 공간에 놓고 가까운 문서를 찾습니다. 실행 전 루트 `.env`의 `EMBED_MODEL`을 현재 AI Core에 배포된 모델명으로 설정하세요. 이 실습은 입력 텍스트의 임베딩과 결과 조회만 수행하며 외부 업무 상태를 변경하지 않습니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 두 텍스트의 의미적 거리 비교 (UI)</h2>
<p style="margin: 7px 0;">아래 UI에 두 텍스트를 입력하고 Compare를 누르세요. embedding 모델은 루트 <code>.env</code>의 <code>EMBED_MODEL</code>을 자동으로 사용합니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>Text A와 Text B에 비교할 텍스트를 입력합니다.</li><li>표시된 <code>Embedding model</code>이 현재 <code>.env</code> 설정과 일치하는지 확인합니다.</li><li>오른쪽 Result에서 <code>cosine_similarity</code>가 1에 가까울수록 두 벡터의 방향이 비슷한지 확인합니다.</li></ol>
<p style="margin: 7px 0;">UI는 조회용입니다. 임베딩 요청만 보내며 업무 데이터나 외부 상태를 수정하지 않습니다.</p>
  </div>
</div>

In [1]:
from pathlib import Path
from time import perf_counter
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
for module_path in (project_root / "1_gen_ai_hub" / "src", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from src.notebook_ui import build_embedding_ui
from src.embeddings.lab_core import compare_embeddings, retrieve_similar_texts

embedding_ui = build_embedding_ui(compare_embeddings)
embedding_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", width="100%", height=680)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>두 텍스트를 입력했을 때 모델명, vector 차원, 비교 결과가 JSON으로 보인다.</li><li>텍스트를 비우면 SDK 호출 전에 입력 오류가 표시된다.</li><li>UI 결과는 검증된 코어 API의 반환값이며, 상태 변경은 발생하지 않는다.</li></ul>
<p style="margin: 7px 0;">문제가 없으면 다음으로 진행하세요. 아래 코드는 같은 코어 API를 Python으로 살펴보는 선택 심화 자료입니다.</p>
  </div>
</div>

<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 테스트된 코어 API로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 호출하는 <code>compare_embeddings()</code>를 사용합니다. SDK 호출, 입력 검증, 벡터 차원 확인, cosine similarity 계산은 모두 코어에 있으므로 노트북이 같은 로직을 다시 구현하지 않습니다.</p>
  </div>
</div>

In [3]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
for module_path in (project_root / "1_gen_ai_hub" / "src", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from lab_common.config import load_settings
from src.embeddings.lab_core import compare_embeddings

settings = load_settings()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 임베딩 생성과 쌍별 비교</h2>
<p style="margin: 7px 0;">텍스트 목록을 한 번에 전달하면 코어가 한 번의 embedding 요청으로 벡터를 받고 모든 텍스트 쌍의 cosine similarity를 반환합니다. 같은 텍스트가 중복되어도 비교 결과가 덮어써지지 않도록 결과는 목록으로 표현합니다.</p>
  </div>
</div>

In [4]:
examples = [
    '결제를 취소하고 싶어요.',
    '주문 환불을 요청합니다.',
    '카드 배송 상태를 알려주세요.',
    '상품 배송 상태를 알려주세요.',
    '환불은 원하지 않아요.',
]
comparison = compare_embeddings(examples, settings)
comparison

{'model': 'amazon--titan-embed-image',
 'dimensions': 1536,
 'texts': ['결제를 취소하고 싶어요.',
  '주문 환불을 요청합니다.',
  '카드 배송 상태를 알려주세요.',
  '상품 배송 상태를 알려주세요.',
  '환불은 원하지 않아요.'],
 'comparisons': [{'left': '결제를 취소하고 싶어요.',
   'right': '주문 환불을 요청합니다.',
   'cosine_similarity': 0.48369077151459716},
  {'left': '결제를 취소하고 싶어요.',
   'right': '카드 배송 상태를 알려주세요.',
   'cosine_similarity': 0.3905932464596482},
  {'left': '결제를 취소하고 싶어요.',
   'right': '상품 배송 상태를 알려주세요.',
   'cosine_similarity': 0.3610340560844885},
  {'left': '결제를 취소하고 싶어요.',
   'right': '환불은 원하지 않아요.',
   'cosine_similarity': 0.3670590521781788},
  {'left': '주문 환불을 요청합니다.',
   'right': '카드 배송 상태를 알려주세요.',
   'cosine_similarity': 0.43461339694927387},
  {'left': '주문 환불을 요청합니다.',
   'right': '상품 배송 상태를 알려주세요.',
   'cosine_similarity': 0.5402266181763037},
  {'left': '주문 환불을 요청합니다.',
   'right': '환불은 원하지 않아요.',
   'cosine_similarity': 0.4079363879632573},
  {'left': '카드 배송 상태를 알려주세요.',
   'right': '상품 배송 상태를 알려주세요.',
   'cosine_similarity': 0.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 반환된 근거 읽기</h2>
<p style="margin: 7px 0;">차원 수와 이미 계산된 비교 결과를 확인합니다. 여기서는 벡터나 cosine 함수를 다시 만들지 않고 코어가 반환한 값을 그대로 읽습니다. 일반적으로 의미가 가까운 쌍의 상대적 순서를 비교하는 것이 유용합니다.</p>
  </div>
</div>

In [5]:
print(f"embedding model: {comparison['model']}")
print(f"vector dimensions: {comparison['dimensions']}")
for item in comparison["comparisons"]:
    print(f"{item['left']} ↔ {item['right']}: {item['cosine_similarity']:.3f}")

embedding model: amazon--titan-embed-image
vector dimensions: 1536
결제를 취소하고 싶어요. ↔ 주문 환불을 요청합니다.: 0.484
결제를 취소하고 싶어요. ↔ 카드 배송 상태를 알려주세요.: 0.391
결제를 취소하고 싶어요. ↔ 상품 배송 상태를 알려주세요.: 0.361
결제를 취소하고 싶어요. ↔ 환불은 원하지 않아요.: 0.367
주문 환불을 요청합니다. ↔ 카드 배송 상태를 알려주세요.: 0.435
주문 환불을 요청합니다. ↔ 상품 배송 상태를 알려주세요.: 0.540
주문 환불을 요청합니다. ↔ 환불은 원하지 않아요.: 0.408
카드 배송 상태를 알려주세요. ↔ 상품 배송 상태를 알려주세요.: 0.726
카드 배송 상태를 알려주세요. ↔ 환불은 원하지 않아요.: 0.274
상품 배송 상태를 알려주세요. ↔ 환불은 원하지 않아요.: 0.303


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">중간 체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>반환값에서 embedding 모델과 vector 차원 수를 확인했다.</li><li>의미가 비슷한 문장뿐 아니라 카드·상품처럼 표면 단어가 비슷한 문장, 환불의 긍정·부정 문장을 함께 비교했다.</li><li>embedding 유사도는 검색 후보를 고르는 신호이며, 부정이나 세부 조건을 완전하게 판별하는 규칙은 아니라는 점을 관찰했다.</li></ul>
  </div>
</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 3</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">질문에서 Top-k FAQ 검색하기</h2>
<p style="margin: 7px 0;">RAG 검색은 질문 하나와 문서 여러 개를 같은 벡터 공간에 놓고, 질문에 가까운 문서를 순위화하는 작업입니다. 코어는 질문과 문서를 한 번의 embedding 요청으로 처리하고, 점수 내림차순의 결과에 문서 위치와 rank를 함께 반환합니다.</p>
  </div>
</div>


In [6]:
query = '결제는 취소했는데 환불금이 아직 안 들어왔어요.'
faq_documents = [
    '결제 취소는 주문 상세 화면에서 요청할 수 있습니다.',
    '환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다.',
    '배송지를 변경하려면 상품 준비 전 고객센터에 문의하세요.',
    '카드 배송 조회는 카드사 앱에서 확인할 수 있습니다.',
    '중복 결제된 주문은 결제 내역과 주문 번호를 함께 보내주세요.',
]
retrieval = retrieve_similar_texts(query, faq_documents, settings, top_k=3)
print(f'질문: {retrieval["query"]}')
for item in retrieval['results']:
    print(f'{item["rank"]}. ({item["cosine_similarity"]:.3f}) {item["document"]}')

질문: 결제는 취소했는데 환불금이 아직 안 들어왔어요.
1. (0.566) 결제 취소는 주문 상세 화면에서 요청할 수 있습니다.
2. (0.373) 중복 결제된 주문은 결제 내역과 주문 번호를 함께 보내주세요.
3. (0.349) 환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 4</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">threshold와 chunk 크기 관찰하기</h2>
<p style="margin: 7px 0;">점수 threshold는 관련 없는 문서를 답변에 넣지 않기 위한 기준입니다. 아래 값은 정답이 아니므로 실행 결과를 보고 조절하세요. 이어서 같은 정책을 짧은 chunk와 긴 문서로 나누어 검색 결과가 어떻게 달라지는지도 확인합니다.</p>
  </div>
</div>


In [7]:
threshold = 0.55  # 실행 결과를 보고 값을 바꿔 보세요.
filtered = retrieve_similar_texts(
    query, faq_documents, settings, top_k=3, min_similarity=threshold
)
print(f'{threshold} 이상 검색 결과: {len(filtered["results"])}건')
for item in filtered['results']:
    print(f'{item["cosine_similarity"]:.3f} | {item["document"]}')

chunk_query = '환불금은 언제 카드에 반영되나요?'
short_chunks = [
    '환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다.',
    '계좌 환불은 등록된 계좌 정보를 확인한 뒤 처리합니다.',
    '배송 중인 상품은 취소가 제한될 수 있습니다.',
]
long_chunk = ['환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다. 계좌 환불은 등록된 계좌 정보를 확인한 뒤 처리합니다. 배송 중인 상품은 취소가 제한될 수 있습니다.']
for label, chunks in [('짧은 chunk', short_chunks), ('긴 문서', long_chunk)]:
    result = retrieve_similar_texts(chunk_query, chunks, settings, top_k=1)
    best = result['results'][0]
    print(f'{label}: {best["cosine_similarity"]:.3f} | {best["document"]}')

0.55 이상 검색 결과: 1건
0.566 | 결제 취소는 주문 상세 화면에서 요청할 수 있습니다.
짧은 chunk: 0.621 | 환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다.
긴 문서: 0.606 | 환불 완료 후 카드사 반영까지 영업일 기준 3~5일이 걸릴 수 있습니다. 계좌 환불은 등록된 계좌 정보를 확인한 뒤 처리합니다. 배송 중인 상품은 취소가 제한될 수 있습니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 실험</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">embedding 모델별 검색 결과 비교</h2>
<p style="margin: 7px 0;">AI Core에 다른 embedding 모델이 RUNNING 상태로 배포되어 있다면 목록에 추가하세요. 차원 수와 응답 시간, Top-1 문서가 모두 같은지 비교합니다. 배포되지 않은 모델명은 호출할 수 없습니다.</p>
  </div>
</div>


In [8]:
models_to_compare = [None]  # None은 .env의 EMBED_MODEL을 사용합니다.
# models_to_compare.append('AI-Core에-배포된-다른-embedding-모델명')

for model_name in models_to_compare:
    started_at = perf_counter()
    outcome = retrieve_similar_texts(query, faq_documents, settings, top_k=1, model=model_name)
    elapsed = perf_counter() - started_at
    best = outcome['results'][0]
    print(f'{outcome["model"]} | {outcome["dimensions"]}차원 | {elapsed:.1f}초')
    print(f'Top-1: {best["cosine_similarity"]:.3f} | {best["document"]}')
    print()

amazon--titan-embed-image | 1024차원 | 1.1초
Top-1: 0.566 | 결제 취소는 주문 상세 화면에서 요청할 수 있습니다.



<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 미션</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">미니 FAQ 검색기 만들기</h2>
<p style="margin: 7px 0;">본인이 정한 업무 도메인에서 FAQ 5개 이상과 질문 2개를 작성하세요. 각 질문에 Top-3를 반환하고, 적절한 threshold를 정해 관련 문서가 없을 때는 빈 결과가 나오도록 설계합니다. 다음 Grounding 실습에서 이 검색 결과에 답변 근거를 붙일 수 있습니다.</p>
  </div>
</div>


In [ ]:
my_faq_documents = [
    # TODO: 같은 업무 도메인의 FAQ를 다섯 개 이상 작성하세요.
]
my_questions = [
    # TODO: FAQ 검색 품질을 확인할 질문 두 개 이상을 작성하세요.
]
my_threshold = 0.0  # TODO: 관찰한 점수를 바탕으로 조정하세요.

# 작성 후 아래 주석을 해제하세요.
# for question in my_questions:
#     result = retrieve_similar_texts(question, my_faq_documents, settings, top_k=3, min_similarity=my_threshold)
#     print(question, result['results'])